In [ ]:
#!pip install --upgrade kafka-python

In [ ]:
import json
import websocket
from kafka import KafkaProducer

# Kafka broker and topic
KAFKA_BROKER = ["broker-1:19092", "broker-2:19092", "broker-3:19092"]   # Adjust to your docker-compose config (e.g., "broker:9092")
KAFKA_TOPIC = "coinbase_feed"

# Initialize Kafka producer
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

# Define WebSocket event handlers
def on_message(ws, message):
    try:
        data = json.loads(message)
        producer.send(KAFKA_TOPIC, value=data)
    except Exception as e:
        print(f"Error processing message: {e}")

def on_error(ws, error):
    print(f"WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    print("WebSocket closed")

def on_open(ws):
    # Subscribe to Coinbase ticker feed (BTC-USD for example)
    subscribe_message = {
        "type": "subscribe",
        "channels": [
            {"name": "ticker", "product_ids": ["BTC-USD", "ETH-USD"]}
        ]
    }
    ws.send(json.dumps(subscribe_message))
    print("Subscribed to Coinbase ticker feed")

if __name__ == "__main__":
    ws = websocket.WebSocketApp(
        "wss://ws-feed.exchange.coinbase.com",
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()


Subscribed to Coinbase ticker feed
